# Disease Prediction Using Machine Learning

## Objective
Build and compare machine-learning classification models for predicting whether a breast tumor is **malignant** or **benign** using the Breast Cancer Wisconsin Diagnostic dataset.

### Models
1. Logistic Regression
2. Random Forest
3. Support Vector Machine (SVM)

### Evaluation metrics
Accuracy, Precision, Recall, F1-score, and ROC-AUC.


## Phase A — Data Preprocessing

The dataset is loaded directly from `scikit-learn`. The target has two classes:
- 0 = malignant
- 1 = benign

Preprocessing includes:
- checking dataset dimensions and missing values
- separating features and target
- stratified train-test split (80/20)
- standardization for Logistic Regression and SVM

Random Forest is tree-based and does not require feature scaling, but using a common pipeline keeps the workflow clear.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report
)


In [ ]:
data = load_breast_cancer(as_frame=True)

X = data.data
y = data.target

print("Dataset shape:", X.shape)
print("Target classes:", dict(enumerate(data.target_names)))
print("\nMissing values:", X.isnull().sum().sum())
X.head()


In [ ]:
print("Class distribution:")
print(y.value_counts().sort_index())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


## Phase B — Model Implementation

Three classification algorithms are trained. Scaling is included inside pipelines for Logistic Regression and SVM to prevent data leakage.


In [ ]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=5000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced"
    ),
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(probability=True, random_state=42))
    ])
}


In [ ]:
results = []
predictions = {}
confusion_matrices = {}

for name, model in models.items():
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    predictions[name] = (y_pred, y_prob)
    confusion_matrices[name] = confusion_matrix(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_df = pd.DataFrame(results).sort_values("F1-score", ascending=False)
results_df


## Phase C — Evaluation

The metrics are calculated on the held-out test set. In this binary classification task, the positive class is **benign (1)** as defined by the scikit-learn dataset.


In [ ]:
for name, (y_pred, _) in predictions.items():
    print("=" * 70)
    print(name)
    print(classification_report(
        y_test, y_pred,
        target_names=data.target_names,
        digits=4
    ))


In [ ]:
# Model comparison plot
plot_df = results_df.set_index("Model")[[
    "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"
]]

ax = plot_df.plot(kind="bar", figsize=(11, 6))
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0.80, 1.01)
ax.grid(axis="y", alpha=0.25)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
# Confusion matrices
for name, cm in confusion_matrices.items():
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm)
    ax.set_title(f"Confusion Matrix - {name}")
    ax.set_xlabel("Predicted Label")
    ax.set_ylabel("True Label")
    ax.set_xticks([0, 1], ["Malignant", "Benign"])
    ax.set_yticks([0, 1], ["Malignant", "Benign"])

    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()


## Findings

The three models achieve strong performance on the held-out test set. The metrics should be interpreted together rather than relying only on accuracy. Precision measures the correctness of positive predictions, recall measures how many actual positive cases are detected, and F1-score balances precision and recall.

These models are educational demonstrations only and **must not be used as a clinical diagnostic system** without extensive external validation, clinical review, and appropriate regulatory safeguards.


## Conclusion

This experiment demonstrates an end-to-end disease-prediction machine-learning workflow:

**Dataset → preprocessing → train/test split → model training → prediction → evaluation → comparison**

The project satisfies the assignment requirements by using a public dataset, implementing three classification algorithms, calculating multiple evaluation metrics, and generating plots for comparison.
